# **Занятие 6. Свёрточные нейронные сети. Классификация цветных изображений**

https://vk.com/lambda_brain

Рассмотрим случай, когда входные данные -- это цветные изображения. Для обработки таких данных были придуманы **свёрточные нейронные сети**, воспользуемся для этого одной из классических моделей.

---

Импортируем нужные пакеты:

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.datasets as dsets
import torchvision.transforms as transforms


Будем обрабатывать стандартный датасет **CIFAR10**, который включает фотографии десяти классов: самолёт, автомобиль, птица, кот, олень, собака, лягушка, лошадь, корабль и грузовик. На каждый класс приходится по 6000 (5000 обучающих и 1000 тестовых) цветных изображений размером 32 * 32 пиксела (и три канала цветности RGB).

In [2]:
input_size = 3*32*32   # Размер изображения в точках * количество цветов
num_classes = 10       # Количество распознающихся классов (10 видов изображений)
n_epochs = 2           # Количество эпох
batch_size = 4         # Размер мини-пакета входных данных
lr = 0.001             # Скорость обучения

Цветные изображения сначала преобразуем в тензоры: **ToTensor()** переводит целочисленные интенсивности 0..255 в диапазон 0..1. Затем **Normalize(mean, std)** вычисляет `(x - mean) / std` для каждого канала. При mean = std = 0.5 значения переходят в диапазон **−1..1**, а не 0..1.

Композицию преобразований задаём через **transforms.Compose()**. Одинаковая нормализация применяется к обучающим, валидационным и тестовым изображениям.

In [3]:
import torchvision.transforms as transforms

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

cifar_trainset = dsets.CIFAR10(root='./data', train=True, download=True, transform=transform)


100%|██████████| 170M/170M [45:40<00:00, 62.2kB/s]   


Аналогично подготовим и тестовый датасет:

In [4]:
cifar_testset = dsets.CIFAR10(root='./data', train=False, download=True, transform=transform)
print(len(cifar_trainset))
print(len(cifar_testset))

50000
10000


Загрузим наши данные:

In [5]:
train_loader = torch.utils.data.DataLoader(dataset=cifar_trainset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=cifar_testset,
                                          batch_size=batch_size,
                                          shuffle=False)

Добавим стандартный шаг обучения:

In [6]:
# импортируем нужные библиотеки
import torch
import numpy as np # всегда пригодится :)
from torch.nn import Linear, Sigmoid

# инициализируем девайс
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# добавляем типовую функцию "шаг обучения"
def make_train_step(model, loss_fn, optimizer):
    def train_step(x, y):
        model.train()
        yhat = model(x)
        loss = loss_fn(yhat, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        return loss.item()
    return train_step

Какую задействовать модель? Вообще, конструирование наиболее эффективных моделей -- это искусство плюс математика. На практике часто можно пользоваться либо простыми стандартными решениями (как было например в случае распознавания рукописных цифр), либо сложными моделями, которые под соответствующие классы задач придумали ведущие специалисты в ML.
В нашем случае мы применим модель LeNet, предложенную Яном ЛеКуном -- она относится к так называемым **свёрточным нейронным сетям (Convolutional Neural Network, CNN)**, хорошо работающим с двумерными изображениями.


---

Подробное описание принципов работы CNN:

https://neurohive.io/ru/tutorial/cnn-na-pytorch/

Главное, обратите внимание на принцип **свёртки** и движущееся окно/**фильтр**.

**Пулинг (pooling)** -- это схожая со скользящим окном техника, когда вместо свёртки по обучаемым весам к значениям в окне применяется некоторая статистическая функция (среднее, максимум, ...). Так, популярная механика тут -- это max pooling. Пулинг выполняет обобщение мелких деталей, устойчиво выделяет некоторый признак независимо от его размера и ориентации.

**Канал** -- это множество фильтров, формирующих оригинальный двумерный вывод. Каналы нередко связываются с цветностью изображения.


In [7]:
import torch.nn as nn
import torch.nn.functional as F


class CifarModel(nn.Module):
    def __init__(self):
        super(CifarModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x



Рассмотрим структуру этой модели подробнее.

По мелким техническим причинам (необходимость выравнивания данных при переходе от свёрточных слоёв к линейным) её удобнее представить в формате классической модели, а не просто композицией слоёв.

Первый слой -- Conv2d(3, 6, 5) с функцией активации ReLU создаёт набор свёрточных фильтров. Первый параметр 3 -- это количество входных каналов изображений, три цвета. Второй параметр 6 -- это количество выходных каналов, третий параметр -- размер фильтра 5x5. На выходе получается 6 фильтров размером 3x5x5 -- и всего модель выделяет (3 * 5 * 5 + 1) * 6 = 456 параметров.
Выходной размер слоя получится 6 * 28 * 28 , где 28 = ((32 - 5) + 1)

Метод MaxPool2d(2,2) -- это реализация max-пулинга (вычисление его аргументов см. по ссылке выше). kernel_size -- размер окна пулинга, stride -- шаг пулинга. Выходной размер слоя таким образом снижаем в два раза: с 6 * 28 * 28 до 6 * 14 * 14.


Далее снова применяется функция Conv2d(6, 16, 5) -- шесть выходных каналов предыдущей функций как входы. Теперь мы применяем 16 фильтров (каждый размером 6 * 5 * 5), и выходной размер слоя будет 16 * 10 * 10, где 10 = (14 - 5) + 1. Всего на уровне обрабатывается (5 * 5 * 6 + 1) * 16 = 2416 параметров.

Следующий max pooling снижает этот выход в два раза -- с 16 * 10 * 10 до 16 * 5 * 5.

И в заключение добавляются три полносвязных слоя Linear. Обратите внимание, что перед ними надо выполнить модификацию структуры передаваемых данных, так как свёрточные слои работают с двумерными изображениями, а линейные -- с векторными наборами. Такое преобразование выполняет x.view(-1, 16 * 5 * 5).

Первый линейный слой из 120 узлов, получает 16 * 5 * 5 входов -- то есть в нём требуется (16 * 5 * 5 + 1) * 120 = 48120 параметров, и далее количество входов-выходов понижается через следующие слои до наших итоговых 10 классов (последний уровень требует (84+1) * 10 = 850 параметров).


In [8]:
from torch import optim, nn

model = CifarModel()
model.to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

train_step = make_train_step(model, loss_fn, optimizer)

for epoch in range(n_epochs):
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        loss = train_step(images, labels)

# print(model.state_dict())
print(loss)

0.902850329875946


In [9]:
model.eval()
with torch.no_grad():
    correct_count = 0
    sample_count = 0
    for image_batch, label_batch in test_loader:
        image_batch = image_batch.to(device)
        label_batch = label_batch.to(device)
        model_output = model(image_batch)
        predictions = model_output.argmax(dim=1)
        sample_count += label_batch.size(0)
        correct_count += (predictions == label_batch).sum().item()

    test_accuracy = 100 * correct_count / sample_count
    print('Точность: {:.2f} %'.format(test_accuracy))

Точность: 55.03 %


Считаем точность -- она получается в районе 55%. Это хороший результат, так как при случайном выборе мы получили бы 10%. Причём её можно существенно повысить, увеличив число эпох обучения. Однако с помощью LeNet обычно с трудом удаётся достигнуть хотя бы 70%, для результата 80% и более лучше использовать например ResNet.


---



В заключение вычислим точность распознавания по каждому из признаков:

In [10]:
class_correct = [0] * num_classes
class_total = [0] * num_classes
model.eval()
with torch.no_grad():
    for image_batch, label_batch in test_loader:
        image_batch = image_batch.to(device)
        label_batch = label_batch.to(device)
        predictions = model(image_batch).argmax(dim=1)
        for class_index in range(num_classes):
            class_mask = label_batch == class_index
            class_total[class_index] += class_mask.sum().item()
            class_correct[class_index] += (
                (predictions == label_batch) & class_mask
            ).sum().item()

for class_index, class_name in enumerate(cifar_testset.classes):
    class_accuracy = 100 * class_correct[class_index] / class_total[class_index]
    print('Точность для {}: {:.2f} %'.format(class_name, class_accuracy))

Точность для airplane: 61.60 %
Точность для automobile: 70.70 %
Точность для bird: 37.20 %
Точность для cat: 33.40 %
Точность для deer: 31.70 %
Точность для dog: 52.30 %
Точность для frog: 73.70 %
Точность для horse: 57.30 %
Точность для ship: 77.00 %
Точность для truck: 55.40 %


## **Задание.**
Увеличение количества эпох (шагов обучения) существенно повышает качество модели. Но есть и другие способы -- поэкспериментируйте например с добавлением новых свёрточных или линейных слоёв, размерами фильтров и их количеством. В процессе экспериментов избегайте переобучения -- когда модель показывает отличные результаты на обучающей выборке, но невысокие на тестовой.


---

В следующем занятии мы займёмся экспериментами с уже обученными моделями.

## Решение задания: варианты архитектуры и ожидаемые результаты

**Ноутбук не запускался; модели не обучались, данные не скачивались.** Далее приведены код для возможного сравнения и предполагаемые выводы, а не результаты проведённых экспериментов. Около 55% в тексте выше — ориентир из условия, не моё измерение. Точности по отдельным классам без обучения неизвестны.

Для сравнения архитектур оставляем одинаковыми разбиение данных, seed, размер пакета, оптимизатор и бюджет обучения. Меняем по одному фактору. Контрольный вариант с двумя эпохами нужен отдельно: иначе нельзя понять, что помогло — архитектура или длительность обучения.

| Вариант | Изменение относительно контроля | Вероятный эффект и причина | Цена / риск |
|---|---|---|---|
| Контроль | Исходная LeNet, 2 эпохи | Ориентир условия — около 55%; наше разбиение и batch size дадут другой результат | Небольшая сеть может недообучиться |
| Больше эпох | Та же LeNet, до 10 эпох | Обычно рост качества до выхода на плато: веса успевают лучше настроиться | Примерно в 5 раз больше обучения без ранней остановки; возможное переобучение |
| Больше каналов | 6 → 16 и 16 → 32, те же фильтры 5×5 | Больше различных признаков: контуры, текстуры, сочетания цветов | Больше вычислений и параметров; выигрыш не гарантирован |
| Меньше фильтры | 3×3 вместо 5×5, каналы 6/16 | Меньше параметров свёрток, лучше сохранение локальных деталей; возможны как рост, так и падение качества | Меньше область обзора; размер карты после свёрток отличается |
| Дополнительная свёртка | Третий слой 16 → 32 с фильтром 3×3 | Сочетания простых признаков в более сложные; вероятен рост после достаточного обучения | На двух эпохах сеть может не успеть обучиться |
| Дополнительный Linear | Слой 84 → 84 перед классификатором | Более сложная граница классов, но ожидаемый выигрыш слабее, чем от улучшения признаков | Дополнительные параметры могут усилить переобучение |

**Вероятный итог:** увеличение числа эпох — первый кандидат на улучшение исходной сети; расширение каналов и дополнительная свёртка могут дать дальнейший прирост. Дополнительный линейный слой сам по себе менее убедителен. Для небольшой CNN после более длительного обучения можно предположить порядок **60–75%**, но это только грубая гипотеза, не доверительный интервал и не обещание. Численные результаты для вариантов оставляем незаполненными до реального запуска.

### Код вариантов моделей

Исходный `CifarModel` сохранён. В новой модели размеры входа первого Linear вычисляются из размеров фильтров; дополнительная свёртка сохраняет размер карты благодаря padding=1. Поэтому изменения архитектуры не приводят к ошибке размеров тензоров.

In [11]:
class ExperimentModel(nn.Module):
    def __init__(self, channel_sizes=(6, 16), kernel_size=5,
                 has_extra_conv=False, has_extra_linear=False):
        super().__init__()
        first_channels, final_channels = channel_sizes
        feature_layers = [
            nn.Conv2d(3, first_channels, kernel_size),
            nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(first_channels, final_channels, kernel_size),
            nn.ReLU(), nn.MaxPool2d(2, 2),
        ]
        spatial_size = (32 - kernel_size + 1) // 2
        spatial_size = (spatial_size - kernel_size + 1) // 2
        if has_extra_conv:
            feature_layers.extend([
                nn.Conv2d(final_channels, 32, 3, padding=1), nn.ReLU(),
            ])
            final_channels = 32
        self.features = nn.Sequential(*feature_layers)
        classifier_layers = [
            nn.Flatten(),
            nn.Linear(final_channels * spatial_size ** 2, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(),
        ]
        if has_extra_linear:
            classifier_layers.extend([nn.Linear(84, 84), nn.ReLU()])
        classifier_layers.append(nn.Linear(84, 10))
        self.classifier = nn.Sequential(*classifier_layers)

    def forward(self, image_batch):
        return self.classifier(self.features(image_batch))


def experiment_plan():
    return [
        {'name': 'baseline_2_epochs', 'epochs': 2, 'options': {}},
        {'name': 'baseline_10_epochs', 'epochs': 10, 'options': {}},
        {'name': 'wider_channels', 'epochs': 10,
         'options': {'channel_sizes': (16, 32)}},
        {'name': 'smaller_kernels', 'epochs': 10,
         'options': {'kernel_size': 3}},
        {'name': 'extra_convolution', 'epochs': 10,
         'options': {'has_extra_conv': True}},
        {'name': 'extra_linear', 'epochs': 10,
         'options': {'has_extra_linear': True}},
    ]

### Честная проверка качества и контроль переобучения

Из 50 000 обучающих изображений выделяем 45 000 для обучения и 5 000 для валидации, с одинаковыми индексами для всех вариантов. Официальные 10 000 тестовых изображений используем **только один раз после выбора модели**. Подбор архитектуры по тесту завышает оценку её качества.

Код ниже сохраняет веса с минимальным validation loss и останавливает обучение после трёх эпох без улучшения. Сравнивать нужно train loss и validation loss: падение первого при росте второго — признак переобучения. Размер пакета 64 и num_workers=0 — разумная стартовая настройка для слабого компьютера, но ускорение и расход памяти зависят от оборудования. Изменение batch size также меняет обучение, поэтому оно одинаково во всех вариантах.

**Все действия находятся в функциях. Вызовы экспериментов не добавлены: эти ячейки сами обучение не запускают.** Исходные учебные ячейки выше содержат скачивание и обучение; команда «Run All» их запустит.

In [12]:
def prepare_loaders(random_seed=42, batch_count=64):
    tensor_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    full_dataset = dsets.CIFAR10(
        root='./data', train=True, download=False, transform=tensor_transform,
    )
    random_generator = torch.Generator().manual_seed(random_seed)
    train_dataset, valid_dataset = torch.utils.data.random_split(
        full_dataset, [45000, 5000], generator=random_generator,
    )
    train_batches = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_count, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(random_seed),
    )
    valid_batches = torch.utils.data.DataLoader(
        valid_dataset, batch_size=batch_count, shuffle=False, num_workers=0,
    )
    return train_batches, valid_batches


def evaluate_model(network_model, data_batches, target_device):
    network_model.eval()
    loss_function = nn.CrossEntropyLoss(reduction='sum')
    total_losses = 0.0
    correct_count = 0
    sample_count = 0
    with torch.no_grad():
        for image_batch, label_batch in data_batches:
            image_batch = image_batch.to(target_device)
            label_batch = label_batch.to(target_device)
            model_output = network_model(image_batch)
            total_losses += loss_function(model_output, label_batch).item()
            correct_count += (
                model_output.argmax(dim=1) == label_batch
            ).sum().item()
            sample_count += label_batch.size(0)
    return {
        'loss': total_losses / sample_count,
        'accuracy': 100 * correct_count / sample_count,
    }


def train_experiment(model_options, epoch_count=10, random_seed=42):
    torch.manual_seed(random_seed)
    target_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    train_batches, valid_batches = prepare_loaders(random_seed)
    network_model = ExperimentModel(**model_options).to(target_device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(network_model.parameters(), lr=0.001)
    best_valid_loss = float('inf')
    best_weights = None
    stalled_epochs = 0
    training_history = []

    for epoch_index in range(epoch_count):
        network_model.train()
        total_losses = 0.0
        sample_count = 0
        correct_count = 0
        for image_batch, label_batch in train_batches:
            image_batch = image_batch.to(target_device)
            label_batch = label_batch.to(target_device)
            optimizer.zero_grad()
            model_output = network_model(image_batch)
            current_loss = loss_function(model_output, label_batch)
            current_loss.backward()
            optimizer.step()
            total_losses += current_loss.item() * label_batch.size(0)
            sample_count += label_batch.size(0)
            correct_count += (
                model_output.detach().argmax(dim=1) == label_batch
            ).sum().item()
        valid_metrics = evaluate_model(network_model, valid_batches, target_device)
        training_history.append({
            'epoch': epoch_index + 1,
            'train_loss': total_losses / sample_count,
            'train_accuracy': 100 * correct_count / sample_count,
            'valid_loss': valid_metrics['loss'],
            'valid_accuracy': valid_metrics['accuracy'],
        })
        if valid_metrics['loss'] < best_valid_loss:
            best_valid_loss = valid_metrics['loss']
            best_weights = {
                weight_name: weight_value.detach().cpu().clone()
                for weight_name, weight_value in network_model.state_dict().items()
            }
            stalled_epochs = 0
        else:
            stalled_epochs += 1
            if stalled_epochs >= 3:
                break

    if best_weights is not None:
        network_model.load_state_dict(best_weights)
    network_model.eval()
    return network_model, training_history

### Какие ещё способы повышения качества применить и почему

1. **Аугментация только обучающих изображений:** `RandomCrop(32, padding=4)` и `RandomHorizontalFlip()` перед `ToTensor()`. Сеть видит разные положения одного объекта и меньше запоминает конкретные картинки. Для CIFAR10 такие преобразования обычно сохраняют класс. Для validation/test оставляем только преобразование в тензор и нормализацию. Важно создать два экземпляра датасета с разными transform и применить к ним сохранённые индексы разбиения: изменение transform у общего датасета затронет обе выборки.
2. **Weight decay:** попробовать `Adam(..., weight_decay=1e-4)`. Ограничение роста весов может уменьшить переобучение. Слишком сильная регуляризация приводит к недообучению; значение выбирается по валидации, а не по тесту.
3. **Dropout:** например, `nn.Dropout(0.2)` после ReLU полносвязного слоя. Случайное отключение части признаков при обучении снижает зависимость от отдельных нейронов. В небольшой исходной LeNet эффект может быть нулевым или отрицательным; полезнее при заметном разрыве между train и validation. При оценке обязательно `model.eval()`.
4. **Понижение learning rate:** `ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)` и `scheduler.step(valid_metrics['loss'])` после валидации. Когда улучшение остановилось, меньшие шаги могут помочь точнее настроить веса. Раннюю остановку стоит сделать менее агрессивной, например patience=4, чтобы дать модели несколько эпох после снижения скорости.
5. **BatchNorm после Conv2d перед ReLU:** может стабилизировать обучение и упростить настройку скорости. Это отдельный эксперимент; результат зависит от размера пакета и архитектуры. Для оценки нужны `eval()` и сохранённые статистики.
6. **Более глубокая сеть с несколькими 3×3 свёртками:** две последовательные 3×3 свёртки без пулинга между ними имеют область обзора 5×5 и дополнительную нелинейность. При одинаковом числе входных, промежуточных и выходных каналов у них меньше весов, чем у одной 5×5 свёртки (18C² против 25C² без bias); при разных каналах экономия не гарантирована. На слабом CPU глубокие модели дороже, поэтому начинаем с компактных вариантов.
7. **ResNet или перенос обучения:** остаточные связи облегчают обучение более глубоких сетей; предобученные признаки могут сократить необходимое обучение. Но модель тяжелее, а перенос с ImageNet на картинки 32×32 требует адаптации входа и классификатора. Это следующий этап, если компактной CNN недостаточно, а не первый эксперимент на слабом компьютере.

**Как выбрать метод:** если обе точности низкие — проверить обучение, увеличить число эпох или выразительность сети. Если train accuracy высокая, а validation accuracy заметно ниже — пробовать аугментацию и регуляризацию, использовать раннюю остановку. Если validation loss вышел на плато — попробовать уменьшить learning rate. После каждого изменения сравнивать одинаковые условия; для надёжных выводов желательно несколько seed, когда появятся вычислительные ресурсы.

**Мой выбор для этой задачи:** сначала исходная LeNet до 10 эпох с валидацией и ранней остановкой, затем аугментация, затем увеличение каналов или третья свёртка. Эти шаги адресуют соответственно недообучение, запоминание данных и нехватку признаков. Не следует складывать ожидаемые прибавки: методы взаимодействуют, а реальный выигрыш выясняется только экспериментом.

Источники: [официальный пример классификации CIFAR10](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html), [примеры преобразований torchvision](https://docs.pytorch.org/vision/stable/auto_examples/transforms/plot_transforms_illustrations.html), [ReduceLROnPlateau](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html).